# Self-Attention with Trainable Weights (Scaled Dot-Product Attention)

This notebook builds on the [vanilla dot-product attention](vanilla-dot-product-attention.ipynb) walkthrough. There, attention scores came directly from the raw token embeddings -- there was nothing to learn, the embeddings themselves decided everything.

Real self-attention layers add three **trainable weight matrices**: $W_q$, $W_k$, and $W_v$. Instead of comparing raw embeddings to each other, each token embedding first gets projected into a **query**, **key**, and **value** vector. Because these projections are *learned* (adjusted during training), the model can gradually figure out the best way to decide "which words should pay attention to which other words" for the task it's being trained on, rather than being stuck with whatever the fixed embeddings happen to look like.

As in the vanilla notebook, every step below is first implemented with explicit `for` loops so you can see exactly what numbers get multiplied and added, and then rewritten with matrix multiplication so we can verify both approaches agree.

## 1. Setup and Token Embeddings

We reuse the same 5-word sentence as before, but this time we distinguish between two sizes:

* **`d_in`** -- the size of each *input* token embedding (how many numbers describe a word before attention).
* **`d_out`** -- the size of the *query/key/value* vectors that come out of the projections.

In a real model these are often the same size, but keeping them different here (`d_in=3`, `d_out=2`) makes it easier to see, just by looking at the shapes, which numbers are "before projection" and which are "after projection".

In [1]:
import torch

torch.manual_seed(42)

sentence = "Hello my name is Ahtesham"
tokens = sentence.split()
num_tokens = len(tokens)
d_in = 3   # dimensionality of the input token embeddings
d_out = 2  # dimensionality of the query/key/value vectors

# Random, non-contextualized embeddings for each token (same idea as before).
token_embeddings = torch.rand(num_tokens, d_in)

print("Tokens:", tokens)
print("\nInput embeddings (token_embeddings):")
print(token_embeddings)
print("\nShape:", token_embeddings.shape)

Tokens: ['Hello', 'my', 'name', 'is', 'Ahtesham']

Input embeddings (token_embeddings):
tensor([[0.8823, 0.9150, 0.3829],
        [0.9593, 0.3904, 0.6009],
        [0.2566, 0.7936, 0.9408],
        [0.1332, 0.9346, 0.5936],
        [0.8694, 0.5677, 0.7411]])

Shape: torch.Size([5, 3])


## 2. The Trainable Weight Matrices $W_q$, $W_k$, $W_v$

Why not just use the raw token embeddings directly, like in the vanilla notebook? Because a single embedding would then have to serve three different jobs at once: asking a question, answering a question, and holding the actual content -- and there's no reason those three jobs should look the same mathematically. So instead we give the model three separate, learnable "lenses" to look at the same embedding through:

* **Query ($W_q$)** -- turns a token into "what am I looking for?" Think of it like typing a search query into a search engine.
* **Key ($W_k$)** -- turns a token into "how would I describe myself, so that others can find me?" Think of it like the index/tags attached to a document in that search engine.
* **Value ($W_v$)** -- turns a token into "what information do I actually hand over once I've been found?" This is the actual content returned by the search, as opposed to the tags used to find it.

Splitting these into three separate matrices means the model can learn, for example, that "asking about X" and "being found for X" don't have to be represented in exactly the same way -- which turns out to make attention much more flexible and effective.

Each weight matrix has shape `(d_in, d_out)`, so multiplying an embedding of shape `(d_in,)` by it produces a vector of shape `(d_out,)`. We keep `requires_grad=False` here purely for illustration -- in a real model these matrices start out random, just like this, and are then *learned* during training so the "lenses" get better and better at their job.

In [2]:
torch.manual_seed(123)

W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key   = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

print("W_query:\n", W_query)
print("\nW_key:\n", W_key)
print("\nW_value:\n", W_value)

W_query:
 Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]])

W_key:
 Parameter containing:
tensor([[0.1366, 0.1025],
        [0.1841, 0.7264],
        [0.3153, 0.6871]])

W_value:
 Parameter containing:
tensor([[0.0756, 0.1966],
        [0.3164, 0.4017],
        [0.1186, 0.8274]])


## 3. Computing Query, Key, and Value Vectors

Let's start with a single token, `"my"` (index 1), to keep things easy to follow.

Multiplying a token's embedding by a weight matrix is called a **linear projection**. All that means in plain terms: each number in the output vector is just a weighted combination (a sum of products) of the numbers in the input embedding, where the weights come from one column of the matrix. That's exactly what the nested loop below does by hand -- for every output position `k`, it walks through every input position `i` and adds up `x[i] * W[i, k]`. The matrix-multiplication shortcut `x @ W_query` computes the exact same sum, just faster and without us having to write the loop ourselves.

In [3]:
query_idx = 1  # "my" -- our example query token
x_query = token_embeddings[query_idx]

# Manual computation: query_manual[k] = sum_i x_query[i] * W_query[i, k]
query_manual = torch.zeros(d_out)
for k in range(d_out):
    total = 0.0
    for i in range(d_in):
        total += x_query[i] * W_query[i, k]
    query_manual[k] = total

query_matmul = x_query @ W_query

print("Query vector (manual for-loops):", query_manual)
print("Query vector (matrix multiplication):", query_matmul)
print("Are they equal?", torch.allclose(query_manual, query_matmul))

Query vector (manual for-loops): tensor([0.4268, 1.2851])
Query vector (matrix multiplication): tensor([0.4268, 1.2851])
Are they equal? True


Now let's generalize this to compute the query, key, **and** value vectors for *every* token at once, again comparing an explicit triple-nested loop against the matrix-multiplication equivalent.

Why do we need a query, a key, *and* a value for every single token, instead of just one for our chosen word? Because in self-attention, **every** word takes a turn being the "current word" -- each token needs to ask its own question (query) about the rest of the sentence, while simultaneously being available to be looked up (key) and to contribute information (value) when some *other* token asks about it.

In [4]:
queries_manual = torch.zeros(num_tokens, d_out)
keys_manual    = torch.zeros(num_tokens, d_out)
values_manual  = torch.zeros(num_tokens, d_out)

for t in range(num_tokens):          # for each token
    for k in range(d_out):           # for each output dimension
        q_sum = k_sum = v_sum = 0.0
        for i in range(d_in):        # for each input dimension
            q_sum += token_embeddings[t, i] * W_query[i, k]
            k_sum += token_embeddings[t, i] * W_key[i, k]
            v_sum += token_embeddings[t, i] * W_value[i, k]
        queries_manual[t, k] = q_sum
        keys_manual[t, k]    = k_sum
        values_manual[t, k]  = v_sum

# Efficient matrix-multiplication equivalent: (N x d_in) @ (d_in x d_out) = (N x d_out)
queries = token_embeddings @ W_query
keys    = token_embeddings @ W_key
values  = token_embeddings @ W_value

print("Queries match:", torch.allclose(queries_manual, queries))
print("Keys match:   ", torch.allclose(keys_manual, keys))
print("Values match: ", torch.allclose(values_manual, values))

print("\nQueries:\n", queries)
print("\nKeys:\n", keys)
print("\nValues:\n", values)

Queries match: True
Keys match:    True
Values match:  True

Queries:
 tensor([[0.5199, 1.4175],
        [0.4268, 1.2851],
        [0.3453, 1.4942],
        [0.3186, 1.2267],
        [0.4551, 1.4822]])

Keys:
 tensor([[0.4096, 1.0182],
        [0.3923, 0.7948],
        [0.4777, 1.2492],
        [0.3773, 1.1004],
        [0.4569, 1.0107]])

Values:
 tensor([[0.4016, 0.8579],
        [0.2673, 0.8427],
        [0.3821, 1.1477],
        [0.3762, 0.8928],
        [0.3333, 1.0122]])


## 4. Computing (Unscaled) Attention Scores

Unlike the vanilla version, attention scores are no longer computed directly from the token embeddings. Instead, they are the dot product of a token's **query** vector with every token's **key** vector: $\omega_{ij} = q^{(i)} \cdot k^{(j)}$.

Why does a dot product measure "relevance"? A dot product multiplies matching positions of two vectors and adds up the results. If two vectors point in a similar direction (their large numbers line up on the same positions), the products are mostly positive and add up to a large value. If they point in unrelated or opposite directions, the products cancel out or stay small. So a **large** dot product between query $i$ and key $j$ means "what token $i$ is looking for lines up well with what token $j$ has to offer" -- in other words, token $j$ is relevant to token $i$.

We first compute the scores for our single example query (`"my"`) against every key, then generalize to the full `(num_tokens x num_tokens)` score matrix, where row `i` holds every attention score for query token `i`.

In [ ]:
# Attention scores for our single query token against every key, via nested loops
attn_scores_single_manual = torch.zeros(num_tokens)
for j in range(num_tokens):
    score = 0.0
    for k in range(d_out):
        score += queries[query_idx, k] * keys[j, k]
    attn_scores_single_manual[j] = score

attn_scores_single_matmul = queries[query_idx] @ keys.T
 
print("Attention scores (manual):", attn_scores_single_manual)
print("Attention scores (matmul):", attn_scores_single_matmul)
print("Equal?", torch.allclose(attn_scores_single_manual, attn_scores_single_matmul))

Attention scores (manual): tensor([1.4833, 1.1888, 1.8092, 1.5752, 1.4938])
Attention scores (matmul): tensor([1.4833, 1.1888, 1.8092, 1.5752, 1.4938])
Equal? True


In [6]:
# Now generalize to the full N x N attention score matrix.
attn_scores_loops = torch.empty(num_tokens, num_tokens)
for i in range(num_tokens):          # each query token
    for j in range(num_tokens):      # each key token
        score = 0.0
        for k in range(d_out):       # dot product over the d_out dimension
            score += queries[i, k] * keys[j, k]
        attn_scores_loops[i, j] = score

attn_scores_matmul = queries @ keys.T

print("Attention scores (loops):")
print(attn_scores_loops)
print("\nAttention scores (matmul):")
print(attn_scores_matmul)
print("\nAre they equal?", torch.allclose(attn_scores_loops, attn_scores_matmul))

# Use the matmul result going forward.
attn_scores = attn_scores_matmul

Attention scores (loops):
tensor([[1.6563, 1.3306, 2.0192, 1.7561, 1.6702],
        [1.4833, 1.1888, 1.8092, 1.5752, 1.4938],
        [1.6628, 1.3231, 2.0316, 1.7746, 1.6680],
        [1.3795, 1.1000, 1.6846, 1.4701, 1.3854],
        [1.6956, 1.3566, 2.0690, 1.8028, 1.7060]])

Attention scores (matmul):
tensor([[1.6563, 1.3306, 2.0192, 1.7561, 1.6702],
        [1.4833, 1.1888, 1.8092, 1.5752, 1.4938],
        [1.6628, 1.3231, 2.0316, 1.7746, 1.6680],
        [1.3795, 1.1000, 1.6846, 1.4701, 1.3854],
        [1.6956, 1.3566, 2.0690, 1.8028, 1.7060]])

Are they equal? True


## 5. Scaling and Normalizing: Scaled Dot-Product Attention

Before turning attention scores into attention weights with `softmax`, we **scale** them by dividing by $\sqrt{d_k}$, where $d_k$ is the length of the key vectors (here, `d_out`). This is why the mechanism is called *scaled* dot-product attention. Let's build up the intuition for why this step exists, one small idea at a time.

**Idea 1: adding up more numbers tends to produce bigger, more spread-out totals.**
A dot product between two vectors of length $d_k$ adds up $d_k$ separate products (one per dimension). Even if each individual product is a small, "reasonably sized" number, adding up more and more of them tends to produce a larger -- and more spread out (higher variance) -- total. This has nothing to do with the vectors being meaningful or not; it's just what happens when you sum more terms. As `d_out` (and `d_k`) grows, the raw dot products naturally grow larger in magnitude, purely as a side effect of dimension size.

**Idea 2: `softmax` reacts very strongly to how big its input numbers are.**
`softmax` turns a list of scores into probabilities that add up to 1, by making bigger scores exponentially more likely than smaller ones. If the scores are already large or very spread apart (as they tend to be when `d_k` is large), `softmax` ends up giving nearly all of the "attention" to whichever score happens to be largest, and pushes everything else down to almost 0. Instead of a smooth blend across several relevant tokens, you get an almost all-or-nothing decision.

**Why is an all-or-nothing decision a problem?**
1. It defeats the purpose of attention. Attention is supposed to let a token softly blend information from *several* relevant tokens -- not just lock onto a single one.
2. It hurts training. When `softmax` outputs are pinned very close to 0 or 1, the "slope" that backpropagation uses to nudge the weights becomes nearly flat (close to zero). With almost no slope to follow, the model's weights barely update -- this problem is known as a **vanishing gradient**, and it can make learning painfully slow or cause it to stall out entirely.

**The fix: divide by $\sqrt{d_k}$.**
There's a neat mathematical fact that rescues us here: when you add up $d_k$ roughly independent random-sized terms (like the products inside a dot product), the *typical size* of the total grows proportionally to $\sqrt{d_k}$, not to $d_k$ itself. So if we divide the dot product by exactly $\sqrt{d_k}$, we cancel out that growth and keep the typical size of the scores roughly the same no matter how big `d_k` is. That keeps `softmax`'s input in a well-behaved range, which in turn keeps its output nicely spread out (instead of all-or-nothing) and keeps the training gradients healthy.

Let's actually see this effect with real numbers before applying it to our own attention scores.

In [ ]:
# A quick, self-contained demonstration of "Idea 1" and "Idea 2" above:
# does the typical size of a dot product really grow with the vector's dimension,
# and does dividing by sqrt(dim) really cancel that growth back out?
import torch

torch.manual_seed(0)

print(f"{'dimension':>10} | {'raw dot-product std':>22} | {'scaled dot-product std':>24}")
for dim in [2, 100, 1000, 10000]:
    a = torch.randn(1000, dim)  # 1000 random vectors of this dimension
    b = torch.randn(1000, dim)  # 1000 more random vectors of this dimension
    dot_products = (a * b).sum(dim=-1)             # 1000 random dot products
    scaled_dot_products = dot_products / dim**0.5  # ... divided by sqrt(dim)

    print(f"{dim:>10} | {dot_products.std().item():>22.2f} | {scaled_dot_products.std().item():>24.2f}")

 dimension |    raw dot-product std |   scaled dot-product std
         2 |                   1.39 |                     0.98
       100 |                   9.68 |                     0.97
      1000 |                  31.48 |                     1.00
     10000 |                  96.25 |                     0.96


As the table above shows: the raw dot-product's typical size (its standard deviation) roughly follows $\sqrt{d_k}$ -- it keeps climbing as the dimension grows. But once we divide by $\sqrt{d_k}$, the typical size settles back down to about `1.0` no matter the dimension. That's the entire trick behind scaling.

In our own example `d_k` is small (just `d_out = 2`), so the effect is subtle here -- but the exact same division keeps things numerically well-behaved, and becomes essential once `d_k` grows into the hundreds or thousands, as it does in real language models.

In [7]:
d_k = keys.shape[-1]

# Manual scaling, element by element.
attn_scores_scaled_manual = torch.empty(num_tokens, num_tokens)
for i in range(num_tokens):
    for j in range(num_tokens):
        attn_scores_scaled_manual[i, j] = attn_scores[i, j] / (d_k ** 0.5)

attn_scores_scaled = attn_scores / d_k**0.5

print("Scaling matches:", torch.allclose(attn_scores_scaled_manual, attn_scores_scaled))

attn_weights = torch.softmax(attn_scores_scaled, dim=-1)

print("\nAttention weights:")
print(attn_weights)
print("\nRow sums (should all be 1.0):", attn_weights.sum(dim=-1))

Scaling matches: True

Attention weights:
tensor([[0.1934, 0.1536, 0.2500, 0.2076, 0.1953],
        [0.1943, 0.1578, 0.2447, 0.2074, 0.1958],
        [0.1934, 0.1521, 0.2510, 0.2093, 0.1941],
        [0.1949, 0.1599, 0.2418, 0.2078, 0.1957],
        [0.1932, 0.1520, 0.2516, 0.2084, 0.1947]])

Row sums (should all be 1.0): tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


## 6. Computing Context Vectors

We now have, for every query token, a full row of attention weights that add up to 1 -- think of it as a "recipe" that says what percentage of each other token's information to mix in. The last step is to actually follow that recipe: multiply each **value** vector by its corresponding weight, and add all of those up. A token that got a high weight contributes a lot to the result; a token with a near-zero weight barely contributes at all.

The result is the **context vector**: a new representation of the token that no longer only describes the word in isolation, but blends in relevant information from the rest of the sentence, weighted by how relevant each other word was judged to be. This is the entire point of self-attention -- turning "my" from a single, context-free embedding into a version of "my" that has absorbed useful context from "Hello", "name", "is", and "Ahtesham" in proportion to how relevant each one is.

In [8]:
context_vectors_manual = torch.zeros(num_tokens, d_out)
for i in range(num_tokens):              # for each query token
    ctx = torch.zeros(d_out)
    for j in range(num_tokens):          # weighted sum over every value vector
        ctx += attn_weights[i, j] * values[j]
    context_vectors_manual[i] = ctx

context_vectors = attn_weights @ values

print("Context vectors match:", torch.allclose(context_vectors_manual, context_vectors))
print("\nContext vectors:")
print(context_vectors)
print("\nShape:", context_vectors.shape)

Context vectors match: True

Context vectors:
tensor([[0.3575, 0.9654],
        [0.3570, 0.9638],
        [0.3577, 0.9656],
        [0.3568, 0.9630],
        [0.3577, 0.9658]])

Shape: torch.Size([5, 2])


## 7. Packaging It Into a Reusable `SelfAttention` Module

Now that we've verified every step by hand, let's package the whole computation -- projecting to queries/keys/values, scoring, scaling, softmax, and weighting the values -- into a single reusable `nn.Module`. This way we can create as many self-attention layers as we like (for different sentences, or stacked inside a bigger network) without repeating all of the steps above every time. This uses `nn.Parameter` directly, matching our manual `W_query` / `W_key` / `W_value` matrices above.

In [9]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(token_embeddings))

tensor([[0.3575, 0.9654],
        [0.3570, 0.9638],
        [0.3577, 0.9656],
        [0.3568, 0.9630],
        [0.3577, 0.9658]], grad_fn=<MmBackward0>)


## 8. A Cleaner Version Using `nn.Linear`

`W_query`, `W_key`, and `W_value` above were just plain matrices that we multiplied by hand. `nn.Linear` layers do exactly the same matrix multiplication internally when their bias is disabled -- so functionally nothing changes. What we gain by using `nn.Linear` is a more effective default weight-initialization scheme (tuned by PyTorch to help networks train well from the start), so this is generally the preferred way to do it in practice, even though the underlying math is identical to what we built by hand.

In [10]:
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(token_embeddings))

tensor([[-0.0661,  0.1392],
        [-0.0657,  0.1398],
        [-0.0643,  0.1422],
        [-0.0666,  0.1384],
        [-0.0646,  0.1417]], grad_fn=<MmBackward0>)


`SelfAttention_v1` and `SelfAttention_v2` give different outputs above because they use different random weight initialization schemes (`nn.Linear` vs. `nn.Parameter(torch.rand(...))`), not because the underlying computation differs.

## 9. Sanity Check: `v1` and `v2` Compute the Same Thing

To prove the two implementations really compute the same thing, we can copy the weights from a `SelfAttention_v2` instance into a `SelfAttention_v1` instance and check that they then produce identical outputs.

The one subtlety: `nn.Linear` stores its weight matrix in **transposed** form (shape `(d_out, d_in)` instead of `(d_in, d_out)`), so we need to transpose it back when assigning to `SelfAttention_v1`'s `nn.Parameter`.

In [11]:
with torch.no_grad():
    sa_v1.W_query.copy_(sa_v2.W_query.weight.T)
    sa_v1.W_key.copy_(sa_v2.W_key.weight.T)
    sa_v1.W_value.copy_(sa_v2.W_value.weight.T)

out_v1 = sa_v1(token_embeddings)
out_v2 = sa_v2(token_embeddings)

print("SelfAttention_v1 output:\n", out_v1)
print("\nSelfAttention_v2 output:\n", out_v2)
print("\nDo v1 and v2 now match?", torch.allclose(out_v1, out_v2, atol=1e-6))

SelfAttention_v1 output:
 tensor([[-0.0661,  0.1392],
        [-0.0657,  0.1398],
        [-0.0643,  0.1422],
        [-0.0666,  0.1384],
        [-0.0646,  0.1417]], grad_fn=<MmBackward0>)

SelfAttention_v2 output:
 tensor([[-0.0661,  0.1392],
        [-0.0657,  0.1398],
        [-0.0643,  0.1422],
        [-0.0666,  0.1384],
        [-0.0646,  0.1417]], grad_fn=<MmBackward0>)

Do v1 and v2 now match? True


## Conclusion

This notebook extended the vanilla dot-product attention walkthrough by introducing trainable weight matrices $W_q$, $W_k$, and $W_v$. Here's the "why" behind each step, in one line:

* **Project to query, key, value** -- so the model can learn separate ways of "asking," "being found," and "answering," instead of forcing one embedding to do all three jobs.
* **Dot product queries with keys** -- because a large dot product between two vectors means they point in similar directions, which we use as a stand-in for "these two tokens are relevant to each other."
* **Scale by $\sqrt{d_k}$** -- because raw dot products naturally grow bigger as the vector dimension grows, which would push softmax into an all-or-nothing decision and stall out training; dividing by $\sqrt{d_k}$ cancels that growth out and keeps things well-behaved.
* **Softmax** -- turns the (now well-scaled) scores into proper attention weights that add up to 1, so they can be read as "what percentage of attention goes to each token."
* **Weighted sum over values** -- blends every token's value vector into the output, in proportion to how relevant it was judged to be, producing a context vector that's aware of the rest of the sentence.

This is exactly the *scaled dot-product self-attention* mechanism used inside Transformer and GPT-style models, and it can be packaged into a reusable `SelfAttention` module built from either raw `nn.Parameter` matrices or `nn.Linear` layers.